# Train DPD Proto-Language Reconstruction Model

**Before running:** Runtime > Change runtime type > **A100 GPU**

Upload `train.pickle`, `dev.pickle`, `test.pickle` to Google Drive at `MyDrive/stang26_data/`

Run cells top to bottom. If you restart the runtime, re-run from cell 1.

In [ ]:
# ============================================================
# CELL 1 — Compatibility patches (MUST run before everything)
# ============================================================

# Python 3.12 removed pkgutil.ImpImporter but old pkg_resources needs it
import pkgutil
if not hasattr(pkgutil, 'ImpImporter'):
    pkgutil.ImpImporter = type('ImpImporter', (), {})

# NumPy 2.0 removed np.Inf etc. but pytorch-lightning 2.0.4 uses them
import numpy as np
for old, new in [('Inf', 'inf'), ('NaN', 'nan'), ('Bool', 'bool_'),
                 ('Int', 'int_'), ('Float', 'float64'), ('Complex', 'complex128'),
                 ('Object', 'object_'), ('Str', 'str_')]:
    if not hasattr(np, old):
        setattr(np, old, getattr(np, new))

print('Patches applied')

In [ ]:
# ============================================================
# CELL 2 — Install dependencies
# ============================================================

# panphon separately with --no-deps to prevent it pulling pyyaml==5.4.1
!pip install -q --no-deps panphon@git+https://github.com/dmort27/panphon.git@6acd3833743a49e63941a0b740ee69eae1dafc1c

# Everything else — no numpy pin (use Colab's numpy 2.x, patched above)
!pip install -q editdistance einops lingpy==2.6.13 lingrex==1.3.0 \
    pytorch-lightning==2.0.4 sacrebleu tabulate toml \
    torchmetrics==0.11.4 scikit-learn scipy \
    newick python-dotenv pandasql unicodecsv regex torchshow

In [ ]:
# ============================================================
# CELL 3 — Verify
# ============================================================
import pytorch_lightning, lingpy, editdistance, panphon, torch
print(f'All imports OK — torch {torch.__version__}, CUDA {torch.cuda.is_available()}')

In [ ]:
# ============================================================
# CELL 4 — Mount Drive, clone DPD, copy data, patch
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!git clone --depth 1 https://github.com/cmu-llab/dpd.git /content/dpd

import shutil, os, pickle

DRIVE_DATA = '/content/drive/MyDrive/stang26_data'
DPD_DATA = '/content/dpd/data/combined'
os.makedirs(DPD_DATA, exist_ok=True)

for split in ['train', 'dev', 'test']:
    src = f'{DRIVE_DATA}/{split}.pickle'
    dst = f'{DPD_DATA}/{split}.pickle'
    assert os.path.exists(src), f'Missing {src}'
    shutil.copy2(src, dst)
    print(f'Copied {split}.pickle')

with open(f'{DPD_DATA}/train.pickle', 'rb') as f:
    langs, data = pickle.load(f)
print(f'{len(data)} train cognate sets, {len(langs)} languages')

# Patch exp.py to accept our dataset
with open('/content/dpd/exp.py', 'r') as f:
    code = f.read()
code = code.replace(
    "choices=[\n    'chinese_wikihan2022', \n    'Nromance_ipa', \n]",
    "choices=[\n    'chinese_wikihan2022', \n    'Nromance_ipa', \n    'combined', \n]"
)
with open('/content/dpd/exp.py', 'w') as f:
    f.write(code)
assert "'combined'" in open('/content/dpd/exp.py').read()

# Dummy .env
with open('/content/dpd/.env', 'w') as f:
    f.write('WANDB_ENTITY = "unused"\nWANDB_PROJECT = "unused"\n')

print('Setup complete')

In [ ]:
# ============================================================
# CELL 5 — Smoke test (5 epochs, ~3 min)
# Stop and check: is loss decreasing? If yes, run cell 6.
# ============================================================
%cd /content/dpd

# DPD authors used batch_size=64 for Trans-DPD on WikiHan (8 langs).
# We have 64 langs so sequences are longer. 64 is safe for A100 40GB.

!python exp.py \
    --nowandb \
    --dataset=combined \
    --architecture=Transformer \
    --strat=bpall_cringe \
    --proportion_labelled=1.0 \
    --exclude_unlabelled \
    --batch_size=64 \
    --test_val_batch_size=64 \
    --max_epochs=5 \
    --check_val_every_n_epoch=1 \
    --min_daughters=1 \
    --d2p_use_lang_separaters=True \
    --p2d_all_lang_summary_only=True \
    --d2p_embedding_dim=256 \
    --d2p_feedforward_dim=512 \
    --d2p_nhead=8 \
    --d2p_num_encoder_layers=2 \
    --d2p_num_decoder_layers=2 \
    --d2p_inference_decode_max_length=30 \
    --d2p_max_len=128 \
    --d2p_dropout_p=0.16 \
    --d2p_recon_loss_weight=0.63 \
    --p2d_embedding_dim=384 \
    --p2d_feedforward_dim=512 \
    --p2d_nhead=8 \
    --p2d_num_encoder_layers=2 \
    --p2d_num_decoder_layers=2 \
    --p2d_inference_decode_max_length=30 \
    --p2d_max_len=128 \
    --p2d_dropout_p=0.34 \
    --p2d_loss_on_gold_weight=0.63 \
    --p2d_loss_on_pred_weight=1.2 \
    --universal_embedding=True \
    --universal_embedding_dim=256 \
    --lr=0.0007 \
    --warmup_epochs=3 \
    --weight_decay=1e-07 \
    --use_xavier_init=True \
    --emb_pred_loss_weight=0.5 \
    --cringe_alpha=0.38 \
    --cringe_k=1 \
    --name smoke_test

In [ ]:
# ============================================================
# CELL 6 — Full training (~20-30 min on A100)
# Only run if smoke test loss was decreasing.
# ============================================================
%cd /content/dpd

!python exp.py \
    --nowandb \
    --logmodel \
    --dataset=combined \
    --architecture=Transformer \
    --strat=bpall_cringe \
    --proportion_labelled=1.0 \
    --exclude_unlabelled \
    --batch_size=64 \
    --test_val_batch_size=128 \
    --max_epochs=200 \
    --check_val_every_n_epoch=5 \
    --early_stopping_patience=10 \
    --min_daughters=1 \
    --d2p_use_lang_separaters=True \
    --p2d_all_lang_summary_only=True \
    --d2p_embedding_dim=384 \
    --d2p_feedforward_dim=512 \
    --d2p_nhead=8 \
    --d2p_num_encoder_layers=2 \
    --d2p_num_decoder_layers=2 \
    --d2p_inference_decode_max_length=30 \
    --d2p_max_len=128 \
    --d2p_dropout_p=0.16 \
    --d2p_recon_loss_weight=0.63 \
    --p2d_embedding_dim=384 \
    --p2d_feedforward_dim=512 \
    --p2d_nhead=8 \
    --p2d_num_encoder_layers=2 \
    --p2d_num_decoder_layers=2 \
    --p2d_inference_decode_max_length=30 \
    --p2d_max_len=128 \
    --p2d_dropout_p=0.34 \
    --p2d_loss_on_gold_weight=0.63 \
    --p2d_loss_on_pred_weight=1.2 \
    --universal_embedding=True \
    --universal_embedding_dim=256 \
    --lr=0.0007 \
    --warmup_epochs=18 \
    --weight_decay=1e-07 \
    --use_xavier_init=True \
    --emb_pred_loss_weight=0.5 \
    --cringe_alpha=0.38 \
    --cringe_k=1 \
    --name ipabier_full

In [ ]:
# ============================================================
# CELL 7 — Save checkpoint to Drive
# ============================================================
import glob, shutil, os

ckpts = sorted(glob.glob('/content/dpd/**/*.ckpt', recursive=True),
               key=os.path.getmtime, reverse=True)

save_dir = '/content/drive/MyDrive/stang26_data/checkpoints'
os.makedirs(save_dir, exist_ok=True)

if ckpts:
    for ckpt in ckpts[:3]:
        dst = f'{save_dir}/{os.path.basename(ckpt)}'
        shutil.copy2(ckpt, dst)
        print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.1f} MB)')
else:
    print('No checkpoints found.')